First we make the necessary imports and set up the seed for reproducibility

In [1]:
import os
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Set global seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("✅ Imports loaded successfully.")

✅ Imports loaded successfully.


We initialize the paths and the target labels

In [2]:
# Define directories
DATA_DIR = '../data/raw' # Contain original .tsv files
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

LABEL_COLS = [
    'Self-direction: thought', 'Self-direction: action', 'Stimulation',
    'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources',
    'Face', 'Security: personal', 'Security: societal', 'Tradition',
    'Conformity: rules', 'Conformity: interpersonal', 'Humility',
    'Benevolence: caring', 'Benevolence: dependability',
    'Universalism: concern', 'Universalism: nature',
    'Universalism: tolerance', 'Universalism: objectivity'
]

We load and merge all the .tsv files into one complete dataset that we will later split

In [3]:
def load_and_merge(split_name):
    """Loads arguments and labels, then merges them on Argument ID."""
    args_path = os.path.join(DATA_DIR, f"arguments-{split_name}.tsv")
    labels_path = os.path.join(DATA_DIR, f"labels-{split_name}.tsv")
    
    # Handle cases where the file might not exist yet during setup
    if not os.path.exists(args_path):
        raise FileNotFoundError(f"Missing {args_path}. Please place TSVs in {DATA_DIR}")
        
    df_args = pd.read_csv(args_path, sep='\t')
    df_labels = pd.read_csv(labels_path, sep='\t')
    return pd.merge(df_args, df_labels, on="Argument ID")

# Load all original splits
df_train_orig = load_and_merge("training")
df_val_orig = load_and_merge("validation")
df_test_orig = load_and_merge("test")

# Combine everything so we can re-split it properly using iterative stratification
all_df = pd.concat([df_train_orig, df_val_orig, df_test_orig], ignore_index=True)

# Ensure all 20 label columns exist (filling missing ones with 0)
for col in LABEL_COLS:
    if col not in all_df.columns:
        all_df[col] = 0

print(f"Total merged dataset size: {len(all_df)} rows.")

Total merged dataset size: 8865 rows.


We preprocess the data in a dual way.
- Keep it raw for transformers (we just concatenate the text) in the column `text_raw`.
- Apply lowercase and lemmatization (winner configuration from assignment 1) for BiLSTM in the column `text_clean`

In [4]:
# 1. RAW TEXT (For Transformers)
# We use [SEP] to explicitly tell BERT/RoBERTa where one section ends and another begins.
all_df['text_raw'] = all_df['Conclusion'] + " [SEP] " + all_df['Stance'] + " [SEP] " + all_df['Premise']

# 2. CLEAN TEXT (For BiLSTM)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_for_lstm(row):
    # Standard concatenation without special tokens
    text = str(row['Conclusion']) + " " + str(row['Stance']) + " " + str(row['Premise'])
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize, remove stopwords, and lemmatize (from Assignment 1 notebook)
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(tokens)

all_df['text_clean'] = all_df.apply(clean_text_for_lstm, axis=1)

print("--- Sample Preprocessing ---")
print("RAW   :", all_df['text_raw'].iloc[0][:120], "...")
print("CLEAN :", all_df['text_clean'].iloc[0][:120], "...")

--- Sample Preprocessing ---
RAW   : We should ban human cloning [SEP] in favor of [SEP] we should ban human cloning as it will only cause huge issues when y ...
CLEAN : ban human cloning favor ban human cloning cause huge issue bunch human running around acting ...


We apply a stratified splitting specific for multilabel problems, making sure we keep an acceptable balance between all labels

In [5]:
# Extract the features and labels we actually need to save
X = all_df[['Argument ID', 'text_raw', 'text_clean']].values
y = all_df[LABEL_COLS].values

# Step 1: Split into 80% Train and 20% Temporary
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_idx, temp_idx in msss1.split(X, y):
    X_train, X_temp = X[train_idx], X[temp_idx]
    y_train, y_temp = y[train_idx], y[temp_idx]

# Step 2: Split the 20% Temporary into 10% Validation and 10% Test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
for val_idx, test_idx in msss2.split(X_temp, y_temp):
    X_val, X_test = X_temp[val_idx], X_temp[test_idx]
    y_val, y_test = y_temp[val_idx], y_temp[test_idx]

# Helper function to reconstruct DataFrames
def reconstruct_df(X_split, y_split):
    df_x = pd.DataFrame(X_split, columns=['Argument ID', 'text_raw', 'text_clean'])
    df_y = pd.DataFrame(y_split, columns=LABEL_COLS)
    return pd.concat([df_x, df_y], axis=1)

train_df = reconstruct_df(X_train, y_train)
val_df = reconstruct_df(X_val, y_val)
test_df = reconstruct_df(X_test, y_test)

print(f"✅ Splits Generated -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

✅ Splits Generated -> Train: 7112 | Val: 860 | Test: 893


We export the data so that subsequent notebooks can load it instantly

In [6]:
# Save to CSV
train_df.to_csv(os.path.join(PROCESSED_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, 'val.csv'), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, 'test.csv'), index=False)

print(f"Data successfully saved to {PROCESSED_DIR}")

Data successfully saved to ../data/processed
